# Quantum Bench

**Logan Mann and Pranav Reddy, University of California, Santa Barbara**

Quantum Bench tests whether an AI agent can diagnose and repair drift in a simulated superconducting quantum computer. This walkthrough covers three environments and reproduces the published error-correction results from recorded data.

[Interactive walkthrough](https://quantum-bench.pages.dev/) · [Repository](https://github.com/itsloganmann/quantum-bench) · [Error-correction paper](https://github.com/itsloganmann/quantum-bench/blob/9b45d297f56e65125d59c788ba3617b9d8867b7c/paper/workshop-20261111/QEC_Drift_Workshop_Paper.pdf)

## Three environments

### 1. Control pulses

The agent gets a nominal transmon model, a control pulse, and calibration traces. It diagnoses the change and returns a revised X-gate pulse. The grader measures gate error on the simulated device. [Draft task specification](https://github.com/pkarakala/qec-drift-bench/blob/7c3804ca5e02063c8e378926e0d2830d945e7161/mismatch/pulse_wrong_model/TASK.md)

### 2. Error correction

The agent gets a Stim surface-code circuit and detector samples from before and after a possible change. It diagnoses the change and returns a detector error model. The grader uses that model with PyMatching and measures logical error recovery on held-out shots. [Draft task specification](https://github.com/pkarakala/qec-drift-bench/blob/7c3804ca5e02063c8e378926e0d2830d945e7161/mismatch/qec_drift_d3/TASK.md)

The original error-correction track asks the agent to write the decoder itself. The worked example and results below use that track. [Scoring specification](https://github.com/itsloganmann/quantum-bench/blob/9b45d297f56e65125d59c788ba3617b9d8867b7c/docs/benchmark-spec-2026-11-01.md)

### 3. Readout

The agent gets labeled calibration measurements and eight batches of unlabeled in-phase and quadrature measurements. It diagnoses the change and submits Gaussian discriminator parameters. The grader measures state assignment error on held-out measurements. [Draft task specification](https://github.com/pkarakala/qec-drift-bench/blob/7c3804ca5e02063c8e378926e0d2830d945e7161/mismatch/readout_drift/TASK.md)

The [three-environment mismatch track](https://github.com/pkarakala/qec-drift-bench/pull/1) is a public draft. A scored Kaggle release still needs a hosted pilot and parity checks. [Kaggle port status](https://github.com/itsloganmann/quantum-bench/blob/main/docs/kaggle-benchmarks-port.md)

## Worked example in error correction

In this recorded run, the agent found a detector whose activity moved over successive blocks of shots. The submitted decoder still fell short of the recovery threshold.

[Public trace excerpt and archived score](https://github.com/itsloganmann/quantum-bench/blob/9b45d297f56e65125d59c788ba3617b9d8867b7c/docs/failure-analysis/moving-fault-example.json)

In [ ]:
record = {'diagnosis_correct': True, 'frozen_gap_closed': 0.10195674562306911, 'frozen_gap_drift': -1.1637717121588087, 'passed_relative': False, 'rule_label': 'plain_matching'}
diagnosis = "correct" if record["diagnosis_correct"] else "incorrect"
result = "pass" if record["passed_relative"] else "fail"
print(f"Diagnosis: {diagnosis}")
print(f"Recovery gap closed: {record['frozen_gap_closed']:.1%}")
print(f"Joint result: {result}")
assert record["diagnosis_correct"] and not record["passed_relative"]


## Error-correction development results

Each available configuration ran on the same 23 development tasks. The joint pass rule requires a correct diagnosis and sufficient decoder recovery on held-out measurements. Ultra enables automatic delegation. N/A marks an unsupported setting.

[Published counts and scores](https://github.com/itsloganmann/quantum-bench/blob/9b45d297f56e65125d59c788ba3617b9d8867b7c/paper/workshop-20261111/evidence/model-pass-rates.json)

In [ ]:
from IPython.display import Markdown, display

records = {'GPT-5.6-Luna': [{'task_count': 23, 'passes': 0, 'pass_percent': 0.0}, {'task_count': 23, 'passes': 0, 'pass_percent': 0.0}, {'task_count': 23, 'passes': 0, 'pass_percent': 0.0}, {'task_count': 23, 'passes': 1, 'pass_percent': 4.3}, {'task_count': 23, 'passes': 3, 'pass_percent': 13.0}, None], 'GPT-5.6-Sol': [{'task_count': 23, 'passes': 4, 'pass_percent': 17.4}, {'task_count': 23, 'passes': 4, 'pass_percent': 17.4}, {'task_count': 23, 'passes': 8, 'pass_percent': 34.8}, {'task_count': 23, 'passes': 7, 'pass_percent': 30.4}, {'task_count': 23, 'passes': 11, 'pass_percent': 47.8}, {'task_count': 23, 'passes': 7, 'pass_percent': 30.4}], 'GPT-5.6-Terra': [{'task_count': 23, 'passes': 0, 'pass_percent': 0.0}, {'task_count': 23, 'passes': 0, 'pass_percent': 0.0}, {'task_count': 23, 'passes': 1, 'pass_percent': 4.3}, {'task_count': 23, 'passes': 2, 'pass_percent': 8.7}, {'task_count': 23, 'passes': 6, 'pass_percent': 26.1}, {'task_count': 23, 'passes': 7, 'pass_percent': 30.4}], 'GPT-6-Astra': [{'task_count': 23, 'passes': 14, 'pass_percent': 60.9}, {'task_count': 23, 'passes': 13, 'pass_percent': 56.5}, {'task_count': 23, 'passes': 13, 'pass_percent': 56.5}, {'task_count': 23, 'passes': 10, 'pass_percent': 43.5}, {'task_count': 23, 'passes': 15, 'pass_percent': 65.2}, {'task_count': 23, 'passes': 14, 'pass_percent': 60.9}]}
names = ["Model", "Low", "Medium", "High", "Very high", "Maximum", "Ultra"]
rows = ["| " + " | ".join(names) + " |", "| " + " | ".join(["---"] * len(names)) + " |"]
for name, runs in records.items():
    values = []
    for run in runs:
        if run is None:
            values.append("N/A")
            continue
        assert run["task_count"] == 23
        pass_rate = 100 * run["passes"] / run["task_count"]
        assert round(pass_rate, 1) == run["pass_percent"]
        values.append(f"{pass_rate:.1f}%")
    rows.append("| " + " | ".join([name, *values]) + " |")
display(Markdown("\n".join(rows)))


## Error-correction replication snapshot

The matched replication covers 85 tasks across 23 configurations, for 1,955 evaluations. This snapshot reports completion counts. The development table above is a separate set of results.

[Latest audited progress](https://github.com/itsloganmann/quantum-bench/blob/main/data/audits/codex-replication-progress.json)

In [ ]:
completed, planned = 1338, 1955
assert 0 <= completed <= planned
print(f"{completed:,} / {planned:,} evaluations completed as of 2026-11-26.")

## Reproduce the error-correction results

Follow the [reproduction guide](https://github.com/itsloganmann/quantum-bench/blob/9b45d297f56e65125d59c788ba3617b9d8867b7c/docs/runbook.md) for simulator setup, task files, and the evaluation harness. The [failure analysis](https://quantum-bench.pages.dev/analysis/) shows the recorded examples and failure breakdown.